# Notebook 02 — CeNGEN Expression Alignment (single-cell, rewritten)

## Goal
Align **single-cell CeNGEN pseudobulked-per-class** expression (CeNGEN thresholded
level 2, "medium") to the 222 Witvliet 2020 adult neurons. Validate the alignment
using Loer & Rand 2022 as ground-truth neurotransmitter classification.

## Why this replaces the previous bulk-based version
- Bulk CeNGEN (Barrett 2022) covers **41 classes** → capped at ~47 mapped neurons.
- Single-cell CeNGEN covers **128 classes** → ~180 mapped neurons (3-4× more).
- Bulk is noisier (pooled sorted cells with cross-talk); single-cell pseudobulk
  is cleaner → transmitter markers now show 100x+ group-level ratios where bulk
  showed ~2-20x.

## Data sources
- Expression: `expression/cengen/thresholded/021821_medium_threshold2.csv`
  (13,669 genes × 128 neuron classes; CeNGEN-recommended threshold level 2)
- NT ground truth: `expression/neurotransmitter/Ce_NTtables_Loer&Rand2022.xlsx`
  (authoritative NT assignments for all 302 hermaphrodite neurons)
- Neurons: from Notebook 01's `data_derived/connectome_adult.npz`

## Preregistered success criteria

**Upgraded from bulk-era targets** (new priors based on knowledge that single-cell
CeNGEN covers 128 classes and Loer & Rand provides 157 ACh / 26 GABA / 71 Glu
ground-truth neurons):

1. **Neuron coverage ≥ 150.** *(Bulk version capped at ~47; single-cell should
   comfortably exceed 150 given 128-class coverage.)*
2. **Genes loaded ≥ 10,000.** *(Thresholded files have 13,669 genes.)*
3. **Symbol resolution ≥ 95%.** *(single-cell file has explicit gene_name column
   for every row.)*
4. **Group-level ACh marker (unc-17):** mean across Witvliet∩CeNGEN-mapped
   cholinergic neurons ≥ 10× mean across Witvliet∩CeNGEN-mapped GABAergic neurons.
   *(Was bulk-only AVAL/RIS check at 2×; now group-level with Loer&Rand N=80+ and
   N=4+, which should be textbook-clean on single-cell.)*
5. **Group-level GABA marker (unc-25):** mean GABAergic ≥ 10× mean cholinergic.
6. **Group-level Glu marker (eat-4/VGLUT):** mean glutamatergic ≥ 10× mean cholinergic.
7. **ASE asymmetry (gcy-7):** ASEL / ASER ≥ 5× *(unchanged — textbook asymmetry).*

## Halting rule
`assert all_pass` — any failure stops the notebook. Bulk fallback is saved as a
sensitivity artifact **only after** single-cell passes.


In [1]:
import sys
from pathlib import Path
_HERE = Path.cwd()
if (_HERE / "lib").is_dir(): sys.path.insert(0, str(_HERE))
elif (_HERE.parent / "lib").is_dir(): sys.path.insert(0, str(_HERE.parent))

from lib.expression import load_expression_singlecell, load_expression, save_expression
from lib.reference import load_nt_reference
from lib.paths import DERIVED

import numpy as np
import pandas as pd

adult = np.load(DERIVED / "connectome_adult.npz", allow_pickle=True)
witvliet_neurons = [str(n) for n in adult["neurons"]]
print(f"Witvliet neurons from Notebook 01: {len(witvliet_neurons)}")

nt = load_nt_reference()
print(f"Loer&Rand NT table loaded ({len(nt.table)} rows)")
print(f"NT breakdown:\n{nt.counts().head(6).to_string()}")


Witvliet neurons from Notebook 01: 222
Loer&Rand NT table loaded (302 rows)
NT breakdown:
nt1
Acetylcholine (ACh)    157
Glutamate (Glu)         71
Unknown                 26
GABA                    26
Dopamine (DA)            8
Serotonin / 5HT          4


## Step 1 — Load single-cell CeNGEN (medium threshold)

In [2]:
expr = load_expression_singlecell(witvliet_neurons, threshold="medium")

m = expr.mapping
mapped = m[m["cengen_class"].notna()]
unmapped = m[m["cengen_class"].isna()]

print(f"Matrix: {expr.tpm.shape}  (neurons x genes)")
print(f"Mapped:   {len(mapped)}/{len(m)}")
print(f"Unmapped: {len(unmapped)}")
print(f"\nRule usage:")
print(mapped["rule_used"].value_counts().to_string())


Matrix: (222, 13669)  (neurons x genes)
Mapped:   179/222
Unmapped: 43

Rule usage:
rule_used
strip_lr            109
strip_positional     40
exact                14
cengen_split_dv      10
cengen_split_lr       6


## Step 2 — Classify unmapped: neurons vs non-neurons

The 222 Witvliet entries include body wall muscle (BWM-*), ring glia (GLR*),
sheath glia (CEPsh*), and neurons whose L/R→ON/OFF assignment is ambiguous (AWCL,
AWCR). These are expected to remain unmapped.

In [3]:
NON_NEURON_PREFIXES = ("BWM-", "GLR", "CEPsh")
AMBIGUOUS = {"AWCL", "AWCR"}

unmapped_names = unmapped["witvliet_name"].tolist()
non_neuron = [n for n in unmapped_names if n.startswith(NON_NEURON_PREFIXES)]
ambiguous = [n for n in unmapped_names if n in AMBIGUOUS]
other = [n for n in unmapped_names if n not in set(non_neuron) | AMBIGUOUS]

print(f"Non-neurons (muscle/glia):         {len(non_neuron)}")
print(f"Ambiguous (AWC L/R→ON/OFF unknown): {len(ambiguous)}")
print(f"Other unmapped:                    {len(other)}")
if other:
    print(f"  Other unmapped items: {other}")


Non-neurons (muscle/glia):         41
Ambiguous (AWC L/R→ON/OFF unknown): 2
Other unmapped:                    0


## Step 3 — Biological sanity: group-level NT markers

In [4]:
def safe_tpm(neuron, gene):
    try:
        v = expr.expression_for(neuron, gene)
        return float(v) if not np.isnan(v) else np.nan
    except KeyError:
        return np.nan

def group_mean(neurons, gene):
    vals = [safe_tpm(n, gene) for n in neurons]
    vals = [v for v in vals if not np.isnan(v)]
    return float(np.mean(vals)) if vals else np.nan

mapped_set = set(mapped["witvliet_name"])
chol_witvliet = [n for n in nt.cholinergic() if n in mapped_set]
gaba_witvliet = [n for n in nt.gabaergic()   if n in mapped_set]
glu_witvliet  = [n for n in nt.glutamatergic() if n in mapped_set]

print(f"Witvliet ∩ CeNGEN-SC ∩ cholinergic:    N={len(chol_witvliet)}")
print(f"Witvliet ∩ CeNGEN-SC ∩ GABAergic:      N={len(gaba_witvliet)}")
print(f"Witvliet ∩ CeNGEN-SC ∩ glutamatergic:  N={len(glu_witvliet)}")

unc17_chol = group_mean(chol_witvliet, "unc-17")
unc17_gaba = group_mean(gaba_witvliet, "unc-17")
unc25_gaba = group_mean(gaba_witvliet, "unc-25")
unc25_chol = group_mean(chol_witvliet, "unc-25")
eat4_glu   = group_mean(glu_witvliet,  "eat-4")
eat4_chol  = group_mean(chol_witvliet, "eat-4")

gcy7_L = safe_tpm("ASEL", "gcy-7")
gcy7_R = safe_tpm("ASER", "gcy-7")

print(f"\nunc-17 (VAChT, chol marker): chol={unc17_chol:.1f}, gaba={unc17_gaba:.1f}")
print(f"unc-25 (GAD, GABA marker):   gaba={unc25_gaba:.1f}, chol={unc25_chol:.1f}")
print(f"eat-4  (VGLUT, glu marker):  glu={eat4_glu:.1f},  chol={eat4_chol:.1f}")
print(f"gcy-7  (ASEL-specific):      ASEL={gcy7_L:.1f}, ASER={gcy7_R:.1f}")


Witvliet ∩ CeNGEN-SC ∩ cholinergic:    N=85
Witvliet ∩ CeNGEN-SC ∩ GABAergic:      N=6
Witvliet ∩ CeNGEN-SC ∩ glutamatergic:  N=52

unc-17 (VAChT, chol marker): chol=218.5, gaba=0.0
unc-25 (GAD, GABA marker):   gaba=24359.7, chol=114.3
eat-4  (VGLUT, glu marker):  glu=2171.2,  chol=14.2
gcy-7  (ASEL-specific):      ASEL=895.6, ASER=0.0


## Step 4 — Evaluate preregistered criteria

In [5]:
def safe_ratio(hi, lo):
    if np.isnan(hi) or np.isnan(lo): return float("nan")
    if lo <= 0: return float("inf") if hi > 0 else 0.0
    return hi / lo

n_mapped = int(mapped.shape[0])
n_genes  = int(expr.n_genes)
symbols = pd.Series(expr.genes_symbol)
resolved = (symbols.notna() & (symbols != "") & (symbols != "nan")
            & ~symbols.astype(str).str.startswith("WBGene"))
frac_resolved = float(resolved.mean())

ratio_chol  = safe_ratio(unc17_chol, unc17_gaba)
ratio_gaba  = safe_ratio(unc25_gaba, unc25_chol)
ratio_glu   = safe_ratio(eat4_glu,   eat4_chol)
ratio_ase   = safe_ratio(gcy7_L, max(gcy7_R, 1.0))

checks = [
    ("1 neuron_coverage >= 150",      n_mapped >= 150,       f"{n_mapped}/{len(witvliet_neurons)}"),
    ("2 genes_loaded >= 10000",       n_genes >= 10000,      f"{n_genes}"),
    ("3 symbol_resolved >= 95%",      frac_resolved >= 0.95, f"{frac_resolved:.2%}"),
    ("4 unc-17 chol/gaba >= 10",      ratio_chol >= 10,      f"{ratio_chol:.2f}x"),
    ("5 unc-25 gaba/chol >= 10",      ratio_gaba >= 10,      f"{ratio_gaba:.2f}x"),
    ("6 eat-4 glu/chol  >= 10",       ratio_glu  >= 10,      f"{ratio_glu:.2f}x"),
    ("7 gcy-7 ASEL/ASER >= 5",        ratio_ase  >= 5,       f"{ratio_ase:.2f}x"),
]

print("PREREGISTERED CRITERIA (single-cell CeNGEN)")
print("=" * 60)
all_pass = True
for label, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {label:42s}  {detail}")
    if not passed: all_pass = False
print("=" * 60)
print(f"ALL CRITERIA PASS: {all_pass}")
assert all_pass, "Halting: single-cell CeNGEN alignment failed criteria."


PREREGISTERED CRITERIA (single-cell CeNGEN)
  [PASS] 1 neuron_coverage >= 150                    179/222
  [PASS] 2 genes_loaded >= 10000                     13669
  [PASS] 3 symbol_resolved >= 95%                    100.00%
  [PASS] 4 unc-17 chol/gaba >= 10                    infx
  [PASS] 5 unc-25 gaba/chol >= 10                    213.14x
  [PASS] 6 eat-4 glu/chol  >= 10                     152.74x
  [PASS] 7 gcy-7 ASEL/ASER >= 5                      895.60x
ALL CRITERIA PASS: True


## Step 5 — Save primary (single-cell) + sensitivity (bulk)

In [6]:
# Primary: single-cell (replaces bulk as canonical)
sc_dir = DERIVED / "expression_sc_medium"
sc_paths = save_expression(expr, sc_dir)

# Also update the canonical `expression_tpm.npz` at top level to point to single-cell
import shutil
for key, p in sc_paths.items():
    dest = DERIVED / p.name
    shutil.copy(p, dest)

print("Primary (single-cell) artifacts:")
for k, p in sc_paths.items():
    print(f"  {k}: {p}  ({p.stat().st_size/1024:.1f} KB)")

# Sensitivity: bulk CeNGEN kept for comparison (never overwritten as primary)
try:
    expr_bulk = load_expression(witvliet_neurons)
    bulk_dir = DERIVED / "expression_bulk"
    bulk_paths = save_expression(expr_bulk, bulk_dir)
    n_bulk = int(expr_bulk.mapping["cengen_class"].notna().sum())
    print(f"\nSensitivity artifact: bulk CeNGEN at {bulk_dir}")
    print(f"  bulk coverage: {n_bulk}/{len(witvliet_neurons)} (compare to SC: {n_mapped}/{len(witvliet_neurons)})")
except Exception as e:
    print(f"(skipping bulk sensitivity: {e})")


Primary (single-cell) artifacts:
  npz: /mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/data_derived/expression_sc_medium/expression_tpm.npz  (4012.3 KB)
  genes_csv: /mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/data_derived/expression_sc_medium/expression_genes.csv  (403.6 KB)
  mapping_csv: /mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/data_derived/expression_sc_medium/expression_neuron_mapping.csv  (4.1 KB)



Sensitivity artifact: bulk CeNGEN at /mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/data_derived/expression_bulk
  bulk coverage: 71/222 (compare to SC: 179/222)


## Step 6 — Save NT reference for downstream notebooks

In [7]:
nt.table.to_csv(DERIVED / "nt_reference_loer_rand_2022.csv", index=False)
print(f"Saved Loer & Rand NT reference to {DERIVED / 'nt_reference_loer_rand_2022.csv'}")
print(f"Columns: {list(nt.table.columns)}")
print(f"Rows (hermaphrodite neurons): {len(nt.table)}")


Saved Loer & Rand NT reference to /mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/data_derived/nt_reference_loer_rand_2022.csv
Columns: ['neuron_class', 'neuron', 'nt1', 'nt2', 'location', 'notes']
Rows (hermaphrodite neurons): 302


## Step 7 — Audit trail & limitations

`data_derived/expression_tpm.npz` is the canonical aligned expression matrix.

**Limitations (documented):**
- Only ~179 of 222 Witvliet neurons have expression data; the rest are muscle,
  glia, or AWC neurons with ambiguous L/R→ON/OFF mapping. Downstream analyses
  must operate on `neurons_with_expression()` or equivalent mask.
- CeNGEN is L4-larval. Witvliet covers L1→adult. Temporal mismatch is real but
  minimized by using the standard CeNGEN reference consistently.
- Single-cell thresholded values are thresholded pseudobulked TPMs; values below
  threshold were set to zero by CeNGEN. Use raw pseudobulk (`L4.all.TPM.raw.qs`)
  if the thresholding is ever a concern — but raw is in R's `qs` format and
  requires pyreadr or an R bridge.

In [8]:
# Final audit table
print("Mapped neurons by CeNGEN class (top 15 classes):")
print(mapped.groupby("cengen_class").size().sort_values(ascending=False).head(15).to_string())
print(f"\nRule-usage totals:")
print(mapped["rule_used"].value_counts().to_string())


Mapped neurons by CeNGEN class (top 15 classes):
cengen_class
IL1       6
OLQ       4
IL2_DV    4
CEP       4
RMD_DV    4
SIB       4
SIA       4
SMD       4
URA       4
URY       4
SMB       4
SAA       4
AFD       2
AIA       2
AIB       2

Rule-usage totals:
rule_used
strip_lr            109
strip_positional     40
exact                14
cengen_split_dv      10
cengen_split_lr       6
